# Model Training and Validation

Trains logistic regression and XGBoost models using expanding-window cross-validation. Evaluates performance separately for US and UK stocks using AUC, Rank IC, and decile spreads.


In [3]:
import os, numpy as np, pandas as pd
from pathlib import Path
from pandas.tseries.offsets import MonthEnd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.preprocessing import StandardScaler
from math import log
import xgboost as xgb
import shap

In [5]:
# CONFIG
INTERIM_PATH = Path("../data/interim")
PROCESSED_PATH = Path("../data/processed")

# TOGGLE THIS to compare v2 (Trends only) vs v3 (Trends + Wikipedia)
DATA_VERSION = "v3"  # Options: "v2" or "v3"

# GEOGRAPHY TAG for SHAP file naming (US vs UK vs Global)
GEO = ""  # Options: "US", "UK", or "" for Global

# Build the file path based on DATA_VERSION
if DATA_VERSION == "v3":
    PREFERRED_FILES = [
        INTERIM_PATH / "features_stocks_v3_1b.parquet",  # 100 stocks + Trends + Wikipedia
        INTERIM_PATH / "features_stocks_v3.parquet",     # fallback (original 10 stocks)
    ]
else:  # v2
    PREFERRED_FILES = [
        INTERIM_PATH / "features_stocks_v2_1b.parquet",  # 100 stocks + Trends only
        INTERIM_PATH / "features_stocks_v2.parquet",     # fallback (original 10 stocks)
    ]

def pick_first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    raise FileNotFoundError(f"None of these files exist: {paths}")

DATA_PATH = pick_first_existing(PREFERRED_FILES)
print(f"Running model on: {DATA_PATH.name} (DATA_VERSION={DATA_VERSION})")

# Build RUN_TAG for SHAP file naming (after DATA_PATH is defined)
DATA_VERSION_FULL = f"{DATA_VERSION}_1b" if "1b" in str(DATA_PATH) else DATA_VERSION
RUN_TAG = f"{DATA_VERSION_FULL}_{GEO}" if GEO else DATA_VERSION_FULL
print(f"RUN_TAG: {RUN_TAG} (for SHAP file naming)")

# STEP 1: USE 12-MONTH TARGETS AS PRIMARY
# Primary targets (12-month horizon)
Y_REG = "Y_REG"  # 12-month excess log return
Y_CLS = "y_up_12m"  # 12-month classification

# Comparison targets (1-month horizon) - for analysis only
Y_REG_1M = "excess_log_next"  # 1-month excess log return
Y_CLS_1M = "y_class_next"  # 1-month classification

# Required baseline columns
ID_COLS   = ["date", "ticker"]
BASE_COLS = ["l_stock","l_bench","ex_log","close","volume"]
STRUCT    = ["country_UK"]
MACRO_LAGS = ["vix_lvl","dxy_lvl","us10y_lvl","dex_usuk","brent_d3m"]

# Load data
df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"]) + MonthEnd(0)
df = df.sort_values(["date","ticker"]).reset_index(drop=True)

print(DATA_PATH.name, df.shape)
print("Tickers:", df["ticker"].nunique(), "| Date range:", df["date"].min(), "→", df["date"].max())
print({c: int(df[c].notna().sum()) for c in ["gt_ma3","gt_z12","gt_spike"] if c in df.columns})



Running model on: features_stocks_v3_1b.parquet (DATA_VERSION=v3)
RUN_TAG: v3_1b (for SHAP file naming)
features_stocks_v3_1b.parquet (24129, 46)
Tickers: 100 | Date range: 2005-02-28 00:00:00 → 2025-09-30 00:00:00
{'gt_ma3': 23881, 'gt_z12': 16705, 'gt_spike': 23881}


In [7]:
# Create labels (next-month excess log return + class) if missing 
if ("excess_log_next" not in df.columns) or ("y_class_next" not in df.columns):
    g = df.groupby("ticker")
    l_stock_next = g["l_stock"].shift(-1)
    l_bench_next = g["l_bench"].shift(-1)
    df["excess_log_next"] = l_stock_next - l_bench_next
    df["y_class_next"] = (df["excess_log_next"] > 0).astype("int8")
    # drop rows where next month label is unavailable (last row per ticker)
    df = df.dropna(subset=["excess_log_next"]).copy()

print("Labels ready:",
      "excess_log_next" in df.columns,
      "y_class_next" in df.columns,
      "| non-null:", int(df["excess_log_next"].notna().sum()))


Labels ready: True True | non-null: 24029


In [9]:
# STEP 1: MERGE 12-MONTH TARGETS 
# Load features
df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"]) + MonthEnd(0)
df = df.sort_values(["date","ticker"]).reset_index(drop=True)

print(f"Loaded features: {len(df):,} rows, {df['ticker'].nunique()} tickers")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

# Merge 12-month targets from notebook 03b
targets_path = PROCESSED_PATH / f"stocks_monthly_targets_splits_{'1b' if '1b' in str(DATA_PATH) else ''}.parquet"
if targets_path.exists():
    targets = pd.read_parquet(targets_path)
    targets["date"] = pd.to_datetime(targets["date"]) + MonthEnd(0)

    print(f"\nMerging 12-month targets from {targets_path.name}")
    print(f"  Targets: {len(targets):,} rows, {targets['ticker'].nunique()} tickers")

    # Merge 12-month targets (primary) and 1-month targets (for comparison)
    before_merge = len(df)
    df = df.merge(
        targets[["date","ticker","Y_REG","y_up_12m","excess_log_next","y_class_next"]],
        on=["date","ticker"],
        how="left",
        validate="one_to_one"
    )
    after_merge = len(df)

    print(f"  After merge: {after_merge:,} rows")
    print(f"  Y_REG (12m) non-null: {df['Y_REG'].notna().sum():,} ({df['Y_REG'].notna().mean():.1%})")
    print(f"  y_up_12m (12m) non-null: {df['y_up_12m'].notna().sum():,} ({df['y_up_12m'].notna().mean():.1%})")

    # Check by country
    if 'country_UK' in df.columns:
        print(f"\n  By country after merge:")
        us_rows = len(df[df['country_UK']==0])
        uk_rows = len(df[df['country_UK']==1])
        us_yreg = df[df['country_UK']==0]['Y_REG'].notna().sum()
        uk_yreg = df[df['country_UK']==1]['Y_REG'].notna().sum()
        print(f"    US (country_UK==0): {us_rows:,} rows, Y_REG non-null: {us_yreg:,}")
        print(f"    UK (country_UK==1): {uk_rows:,} rows, Y_REG non-null: {uk_yreg:,}")
else:
    print(f"Targets file not found: {targets_path}")
    print("  Will create 1-month targets as fallback")
    g = df.groupby("ticker")
    l_stock_next = g["l_stock"].shift(-1)
    l_bench_next = g["l_bench"].shift(-1)
    df[Y_REG_1M] = (l_stock_next - l_bench_next)
    df[Y_CLS_1M] = (df[Y_REG_1M] > 0).astype("int8")
    df[Y_REG] = df[Y_REG_1M]
    df[Y_CLS] = df[Y_CLS_1M]

# Remove rows without 12-month target (need full 12-month window)
before_drop = len(df)
df = df.dropna(subset=[Y_REG, Y_CLS]).copy()
after_drop = len(df)
print(f"\nAfter dropping NaN targets: {before_drop:,} → {after_drop:,} rows (dropped {before_drop - after_drop:,})")

# Final check by country
if 'country_UK' in df.columns:
    print(f"\nFinal data by country:")
    us_final = len(df[df['country_UK']==0])
    uk_final = len(df[df['country_UK']==1])
    us_tickers = df[df['country_UK']==0]['ticker'].nunique()
    uk_tickers = df[df['country_UK']==1]['ticker'].nunique()
    print(f"  US (country_UK==0): {us_final:,} rows, {us_tickers} tickers")
    print(f"  UK (country_UK==1): {uk_final:,} rows, {uk_tickers} tickers")


Loaded features: 24,129 rows, 100 tickers
Date range: 2005-02-28 00:00:00 to 2025-09-30 00:00:00

Merging 12-month targets from stocks_monthly_targets_splits_1b.parquet
  Targets: 22,929 rows, 100 tickers
  After merge: 24,129 rows
  Y_REG (12m) non-null: 22,929 (95.0%)
  y_up_12m (12m) non-null: 22,929 (95.0%)

  By country after merge:
    US (country_UK==0): 12,069 rows, Y_REG non-null: 11,469
    UK (country_UK==1): 12,060 rows, Y_REG non-null: 11,460

After dropping NaN targets: 24,129 → 22,929 rows (dropped 1,200)

Final data by country:
  US (country_UK==0): 11,469 rows, 50 tickers
  UK (country_UK==1): 11,460 rows, 50 tickers


In [11]:
df["date"] = pd.to_datetime(df["date"]) + MonthEnd(0)

REQUIRED_MACRO = ["vix_lvl","dxy_lvl","us10y_lvl","brent_d3m","dex_usuk"]

# If any missing, merge from macro_monthly.parquet
missing = [c for c in REQUIRED_MACRO if c not in df.columns]
if missing:
    macro_p = INTERIM_PATH / "macro_monthly.parquet"
    assert macro_p.exists(), f"Missing {macro_p}; run Notebook 04 to create it."
    macro = pd.read_parquet(macro_p)
    macro["date"] = pd.to_datetime(macro["date"]) + MonthEnd(0)
    df = df.merge(macro, on="date", how="left", validate="many_to_one")
    print("Merged missing macros:", missing)

# Alias map (promote alternates to canonical)
ALIAS_MAP = {
    "vix_lvl":   ["vix","VIX","VIXCLS","vix_level"],
    "dxy_lvl":   ["dxy","DTWEXBGS","broad_usd"],
    "us10y_lvl": ["us10y","DGS10","us10y_yield"],
    "brent_d3m": ["brent_d3m","brent_3m","brent_chg3m","brent_delta3m"],
    "dex_usuk":  ["DEXUSUK","gbpusd","GBPUSD"],
}
for canon, alts in ALIAS_MAP.items():
    if canon not in df.columns:
        for a in alts:
            if a in df.columns:
                df[canon] = df[a]
                print(f"Aliased {a} → {canon}")
                break

# Coalesce any _x/_y to canonical
def coalesce_to(df, base):
    cols = [c for c in df.columns if c == base or c.startswith(base + "_")]
    if not cols:
        return df
    df[base] = pd.concat(
        [df[c] for c in cols if c != base] + ([df[base]] if base in cols else []),
        axis=1
    ).bfill(axis=1).iloc[:,0]
    for c in cols:
        if c != base:
            df.drop(columns=c, inplace=True, errors="ignore")
    return df

for base in REQUIRED_MACRO:
    df = coalesce_to(df, base)

print("After coalesce →")
for c in REQUIRED_MACRO:
    print(f"{c:10s}", c in df.columns, "| non-null%:", df[c].notna().mean() if c in df.columns else "—")

# Ensure no infinities
df = df.replace([np.inf, -np.inf], np.nan)


After coalesce →
vix_lvl    True | non-null%: 1.0
dxy_lvl    True | non-null%: 0.9539447860787649
us10y_lvl  True | non-null%: 1.0
brent_d3m  True | non-null%: 1.0
dex_usuk   True | non-null%: 1.0


In [13]:
years = df["date"].dt.year
FOLDS = [
    {"train_end": 2016, "val": 2017},
    {"train_end": 2017, "val": 2018},
    {"train_end": 2018, "val": 2019},
    {"train_end": 2019, "val": 2020},
    {"train_end": 2020, "val": 2021},
    {"train_end": 2021, "val": 2022},
]
TEST_YEARS = [2023, 2024]  # for later use if you add a hold-out test


In [15]:
def roll_quantile(series, win, q):
    return series.rolling(win, min_periods=win).quantile(q)

def z12(series):
    mu = series.rolling(12, min_periods=12).mean()
    sd = series.rolling(12, min_periods=12).std()
    return (series - mu) / sd

def add_regime_flags_per_fold(dtr, dva, win=60):
    # TRAIN thresholds (past-only on train)
    vix_p75_tr     = roll_quantile(dtr["vix_lvl"], win, 0.75)
    abs_br_p80_tr  = roll_quantile(dtr["brent_d3m"].abs(), win, 0.80)

    th = (pd.DataFrame({
            "date": dtr["date"],
            "vix_p75": vix_p75_tr,
            "abs_br_p80": abs_br_p80_tr,
        })
        .groupby("date", as_index=False).tail(1))
    dtr = dtr.merge(th, on="date", how="left")
    dva = dva.merge(th, on="date", how="left")

    for d in (dtr, dva):
        d["high_vix"]      = (d["vix_lvl"] >= d["vix_p75"]).astype("int8")
        d["oil_shock"]     = (d["brent_d3m"].abs() >= d["abs_br_p80"]).astype("int8")
        d["strong_dollar"] = (z12(d["dxy_lvl"])   >  1.0).astype("int8")
        d["strong_gbp"]    = (z12(d["dex_usuk"])  >  1.0).astype("int8")

        d["bear_tape"] = (
            d.groupby("ticker")["l_bench"]
             .shift(1).rolling(12, min_periods=12).sum() < 0
        ).astype("int8")

        ben_vol3 = d["l_bench"].shift(1).rolling(3, min_periods=3).std()
        vol_p75  = roll_quantile(ben_vol3, win, 0.75)
        d["vol_regime"] = (ben_vol3 >= vol_p75).astype("int8")

        d["strong_gbp_uk"] = (d.get("strong_gbp", 0) * d.get("country_UK", 0)).astype("int8")

        for c in ["high_vix","oil_shock","strong_dollar","strong_gbp","bear_tape","vol_regime","strong_gbp_uk"]:
            d[c] = d[c].shift(1)

    # Debug: Check regime flag creation
    print(f"Regime flags created - dtr: {[c for c in ['high_vix','oil_shock','strong_dollar','strong_gbp','bear_tape','vol_regime','strong_gbp_uk'] if c in dtr.columns]}")
    print(f"Regime flags non-null counts - dtr: {dtr[['high_vix','oil_shock','strong_dollar','strong_gbp','bear_tape','vol_regime','strong_gbp_uk']].notna().sum().to_dict()}")
    print(f"Regime flags non-null counts - dva: {dva[['high_vix','oil_shock','strong_dollar','strong_gbp','bear_tape','vol_regime','strong_gbp_uk']].notna().sum().to_dict()}")

    # FIXED: Only drop NaNs for flags that actually exist and have data
    regime_cols = ["high_vix","oil_shock","strong_dollar","strong_gbp","bear_tape","vol_regime","strong_gbp_uk"]
    existing_regime_cols = [c for c in regime_cols if c in dtr.columns and dtr[c].notna().any()]

    if existing_regime_cols:
        dtr = dtr.dropna(subset=existing_regime_cols)
        dva = dva.dropna(subset=existing_regime_cols)
        print(f"Dropped NaNs for regime flags: {existing_regime_cols}")
    else:
        print("No regime flags to drop NaNs for")

    return dtr, dva


In [17]:
def recency_weights(dates, half_life_months=24):
    d = pd.to_datetime(dates, errors="coerce")
    n = len(d)
    if n == 0:
        return np.array([], dtype=float)
    if d.notna().sum() == 0:
        return np.ones(n, dtype=float)

    m0 = d[d.notna()].min().to_period("M")
    periods = d.dt.to_period("M")
    age_months = []
    for p in periods:
        if pd.isna(p):
            age_months.append(np.nan)
        else:
            age_months.append((p - m0).n)

    age = pd.Series(age_months, index=dates.index)
    fill_val = int(age[age.notna()].median()) if age.notna().any() else 0
    age = age.fillna(fill_val).astype(int)

    w = np.exp(-log(2) * (age / half_life_months))
    return w / w.mean()


In [19]:
# feature family mapper (for SHAP family bar chart) 
def feature_family(name: str) -> str:
    n = name.lower()
    if n.startswith(("mom_log","rev_")):       return "momentum"
    if n.startswith(("vol_", "tracking_error")): return "volatility"
    if n in {"near_high_12","dist_sma_12","drawdown_12"}: return "price_positioning"
    if n in {"vix_lvl","dxy_lvl","us10y_lvl","dex_usuk","brent_d3m"}: return "macro"
    if n in {"high_vix","oil_shock","strong_dollar","strong_gbp","bear_tape","vol_regime","strong_gbp_uk"}: return "regime"
    if n.startswith("gt_"):                    return "google_trends"
    if n.startswith("pv_"):                    return "pageviews"
    if n.startswith("sec_") or n=="country_uk":return "structure"
    return "other"

def build_feature_list(df):
    exclude = set(ID_COLS + [Y_REG, Y_CLS])

    # FIXED: Use more robust features that don't require 36 months of data
    core_price = [
        "mom_log_3","mom_log_6","mom_log_12","rev_1m","vol_6","vol_12",
        "sharpe_12","near_high_12","dist_sma_12","drawdown_12",
        "tracking_error_12",  # Keep this one (12 months)
        # REMOVED: "beta_36","up_cap_36","down_cap_36","up_down_cap_ratio_36" (require 36 months)
    ]

    # liquidity column may be named differently
    if "liquidity" in df.columns:
        core_price.append("liquidity")
    elif "liquidity_12m" in df.columns:
        core_price.append("liquidity_12m")

    sector_dummies = [c for c in df.columns if c.startswith("sec_")]
    structure = ["country_UK"] + sector_dummies

    # Geo-specific macro features (Option 1: separate US/UK features)
    # Detect geography from the dataframe
    if "country_UK" in df.columns and len(df) > 0:
        uk_pct = df["country_UK"].mean() if df["country_UK"].notna().any() else 0.5
        if uk_pct == 0.0:
            # US-only: US-specific macro features
            macro_lvls = ["vix_lvl", "dxy_lvl", "us10y_lvl", "brent_d3m"]
            geo_label = "US"
        elif uk_pct == 1.0:
            # UK-only: UK-specific macro features
            macro_lvls = ["uk10y_lvl", "dex_usuk", "brent_d3m"]
            geo_label = "UK"
        else:
            # Mixed (global): use all features for backward compatibility
            macro_lvls = ["vix_lvl", "dxy_lvl", "us10y_lvl", "uk10y_lvl", "dex_usuk", "brent_d3m"]
            geo_label = "MIXED"
        print(f"[GEO-FEATURES] Detected: {geo_label} (uk_pct={uk_pct:.3f}) → Using {len(macro_lvls)} macro features: {macro_lvls}")
    else:
        # Fallback: use all features if country_UK not available
        macro_lvls = ["vix_lvl", "dxy_lvl", "us10y_lvl", "uk10y_lvl", "dex_usuk", "brent_d3m"]
        print(f"[GEO-FEATURES] country_UK not found → Using all {len(macro_lvls)} macro features")

    regime = ["high_vix","oil_shock","strong_dollar","strong_gbp","bear_tape","vol_regime","strong_gbp_uk"]
    trends = [c for c in ["gt_ma3","gt_z12","gt_spike","pv_ma3","pv_z12","pv_spike"] if c in df.columns]

    # Get all candidate features
    all_candidates = core_price + structure + macro_lvls + regime + trends
    feat = [c for c in all_candidates if c in df.columns and c not in exclude]
    feat = list(dict.fromkeys(feat))  # de-dupe preserve order

    # FIXED: Filter out features with too many NaNs (>50% missing)
    if len(feat) > 0:
        nan_pct = df[feat].isnull().mean()
        good_feat = [c for c in feat if nan_pct[c] <= 0.5]  # Keep features with ≤50% NaNs
        print(f"Feature filtering: {len(feat)} → {len(good_feat)} (removed {len(feat)-len(good_feat)} high-NaN features)")
        if len(feat) - len(good_feat) > 0:
            removed = [c for c in feat if c not in good_feat]
            print(f"Removed features: {removed[:5]}{'...' if len(removed) > 5 else ''}")
        feat = good_feat

    return feat


In [21]:
print("n cols:", len(df.columns))
print(sorted(df.columns)[:60], "...")
for c in ["vix_lvl","dxy_lvl","us10y_lvl","brent_d3m","dex_usuk"]:
    print(f"{c:12s} present? {c in df.columns}  non-null%:",
          df[c].notna().mean() if c in df.columns else "—")


n cols: 50
['Y_REG', 'beta_36', 'brent_d3m', 'close', 'country_UK', 'date', 'dex_usuk', 'dist_sma_12', 'drawdown_12', 'dxy_lvl', 'ex_log', 'excess_log_next', 'gt_ma3', 'gt_spike', 'gt_z12', 'l_bench', 'l_stock', 'liquidity_12m', 'mom_log_12', 'mom_log_3', 'mom_log_6', 'near_high_12', 'pv_ma3', 'pv_spike', 'pv_z12', 'rev_1m', 'sec_CommServices', 'sec_Discretionary', 'sec_Energy', 'sec_Financials', 'sec_Healthcare', 'sec_Industrials', 'sec_Materials', 'sec_RealEstate', 'sec_Staples', 'sec_Tech', 'sec_Utilities', 'sector_rel_mom12', 'sharpe_12', 'ticker', 'tracking_error_12', 'uk10y_lvl', 'up_cap_36', 'us10y_lvl', 'vix_lvl', 'vol_12', 'vol_6', 'volume', 'y_class_next', 'y_up_12m'] ...
vix_lvl      present? True  non-null%: 1.0
dxy_lvl      present? True  non-null%: 0.9539447860787649
us10y_lvl    present? True  non-null%: 1.0
brent_d3m    present? True  non-null%: 1.0
dex_usuk     present? True  non-null%: 1.0


In [23]:
def eval_rank_ic(df_sub, y_reg_col, y_pred):
    m = df_sub.assign(pred=y_pred).groupby(df_sub["date"].dt.to_period("M"))
    ics = []
    for _, g in m:
        if g[y_reg_col].nunique() > 1 and g["pred"].nunique() > 1:
            ics.append(g[[y_reg_col,"pred"]].corr(method="spearman").iloc[0,1])
    return np.nanmean(ics) if ics else np.nan

def decile_spread(df_sub, y_reg_col, y_score):
    d = df_sub.copy()
    d["score"] = y_score
    rets = []
    for dt, g in d.groupby(d["date"].dt.to_period("M")):
        if len(g) < 10:
            continue
        q = np.quantile(g["score"], [0.1, 0.9])
        long = g[g["score"] >= q[1]][y_reg_col].mean()
        short = g[g["score"] <= q[0]][y_reg_col].mean()
        rets.append(long - short)
    if not rets:
        return np.nan, np.nan, 0
    arr = np.asarray(rets)
    t = arr.mean() / (arr.std(ddof=1) / np.sqrt(len(arr))) if arr.std(ddof=1) > 0 else np.nan
    return arr.mean(), t, len(arr)

# Sector helpers and sector-neutral rank-IC 

def infer_sector_label(df_row):
    """
    We turn one-hot columns like sec_Tech, sec_Healthcare... into a single label.
    If none exist, label as 'Unknown'.
    """
    sec_cols = [c for c in df_row.index if c.startswith("sec_")]
    if not sec_cols:
        return "Unknown"
    # pick the sec_* with value==1 ; if tie/missing, fall back to max
    vals = [(c, df_row.get(c, 0)) for c in sec_cols]
    best = max(vals, key=lambda x: x[1])[0] if vals else None
    if best is None or df_row.get(best, 0) <= 0:
        return "Unknown"
    # sec_Tech -> Tech
    return best.replace("sec_", "")

def attach_sector_labels(df):
    if "sector_eval" in df.columns:
        return df
    sec_cols = [c for c in df.columns if c.startswith("sec_")]
    if not sec_cols:
        df["sector_eval"] = "Unknown"
        return df
    df["sector_eval"] = df.apply(infer_sector_label, axis=1)
    return df

def sector_neutral_rank_ic(df_sub, y_reg_col, y_pred, sector_col="sector_eval"):
    """
    For each month: within each sector, rank by prediction, then compute Spearman
    correlation to next-month excess return. Average across sectors & months.
    """
    d = df_sub.copy()
    d["pred"] = y_pred
    out_ics = []
    # group by month, then sector
    for (month, sector), g in d.groupby([d["date"].dt.to_period("M"), sector_col]):
        if g[y_reg_col].nunique() > 1 and g["pred"].nunique() > 1 and len(g) >= 5:
            ic = g[[y_reg_col, "pred"]].corr(method="spearman").iloc[0,1]
            out_ics.append(ic)
    return np.nanmean(out_ics) if out_ics else np.nan

def run_fold(train_end, val_year):
    tr = df[years <= train_end].copy()
    va = df[years == val_year].copy()

    print(f"Fold {train_end}-{val_year}: Before regime flags - tr={len(tr)}, va={len(va)}")

    # add fold-aware regime flags
    tr, va = add_regime_flags_per_fold(tr, va, win=60)

    print(f"Fold {train_end}-{val_year}: After regime flags - tr={len(tr)}, va={len(va)}")

    FEATS = build_feature_list(tr)
    print(f"Features found: {len(FEATS)}")
    print(f"Feature list: {FEATS[:10]}...")  # Show first 10 features

    if not FEATS:
        print(f"[skip] No features present for train_end={train_end}, val_year={val_year}")
        return ({
            "train_end": train_end, "val_year": val_year,
            "logit_auc": np.nan, "xgb_auc": np.nan,
            "logit_rankIC": np.nan, "xgb_rankIC": np.nan,
            "logit_rankIC_sec": np.nan, "xgb_rankIC_sec": np.nan,
            "logit_spread": np.nan, "xgb_spread": np.nan,
            "months": 0, "n_train": 0, "n_val": 0, "n_feats": 0
        }, FEATS)

    # Build matrices
    print(f"Before dropna - tr shape: {tr[FEATS].shape}, va shape: {va[FEATS].shape}")

    # Check NaN counts in features
    tr_nans = tr[FEATS].isnull().sum()
    va_nans = va[FEATS].isnull().sum()
    print(f"Train NaN counts (top 10): {tr_nans.nlargest(10).to_dict()}")
    print(f"Val NaN counts (top 10): {va_nans.nlargest(10).to_dict()}")

    Xtr = tr[FEATS].dropna()
    ytr = tr.loc[Xtr.index, Y_CLS]
    wtr = recency_weights(tr.loc[Xtr.index, "date"], half_life_months=24)

    Xva = va[FEATS].dropna()
    yva = va.loc[Xva.index, Y_CLS]
    yva_reg = va.loc[Xva.index, Y_REG]

    print(f"After dropna - Xtr shape: {Xtr.shape}, Xva shape: {Xva.shape}")

    if Xtr.empty or Xva.empty:
        print(f"[skip] train_end={train_end}, val_year={val_year} (n_train={len(Xtr)}, n_val={len(Xva)})")
        return ({
            "train_end": train_end, "val_year": val_year,
            "logit_auc": np.nan, "xgb_auc": np.nan,
            "logit_rankIC": np.nan, "xgb_rankIC": np.nan,
            "logit_rankIC_sec": np.nan, "xgb_rankIC_sec": np.nan,
            "logit_spread": np.nan, "xgb_spread": np.nan,
            "months": 0, "n_train": len(Xtr), "n_val": len(Xva),
            "n_feats": len(FEATS)
        }, FEATS)

    # Scale for logistic
    scaler = StandardScaler(with_mean=True, with_std=True)
    Xtr_s = pd.DataFrame(scaler.fit_transform(Xtr), index=Xtr.index, columns=Xtr.columns)
    Xva_s = pd.DataFrame(scaler.transform(Xva),    index=Xva.index, columns=Xva.columns)

    # Logistic (baseline)
    logit = LogisticRegression(max_iter=2000, C=1.0, solver="lbfgs")
    logit.fit(Xtr_s, ytr, sample_weight=wtr)
    p_val = logit.predict_proba(Xva_s)[:,1]
    auc_l = roc_auc_score(yva, p_val)

    # threshold tuning on validation (balanced accuracy)
    grid = np.linspace(0.2, 0.8, 121)
    best_t, best_bal = 0.5, -1
    for t in grid:
        bal = balanced_accuracy_score(yva, (p_val >= t).astype(int))
        if bal > best_bal:
            best_bal, best_t = bal, t

    ic_l   = eval_rank_ic(va.loc[Xva.index], Y_REG, p_val)
    sprd_l, tstat_l, n_m = decile_spread(va.loc[Xva.index], Y_REG, p_val)

    # NEW: sector-neutral rank-IC for Logistic
    va_log = va.loc[Xva.index].copy()
    va_log = attach_sector_labels(va_log)
    ic_l_sec = sector_neutral_rank_ic(va_log, Y_REG, p_val, sector_col="sector_eval")

    # XGBoost
    xgbc = xgb.XGBClassifier(
        max_depth=3, n_estimators=700, learning_rate=0.07,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        reg_lambda=1.0, tree_method="hist", n_jobs=-1, random_state=42
    )
    xgbc.fit(Xtr, ytr, sample_weight=wtr, eval_set=[(Xva, yva)], verbose=False)
    px_val = xgbc.predict_proba(Xva)[:,1]
    auc_x  = roc_auc_score(yva, px_val)

    best_tx, best_bax = 0.5, -1
    for t in grid:
        bal = balanced_accuracy_score(yva, (px_val >= t).astype(int))
        if bal > best_bax:
            best_bax, best_tx = bal, t

    ic_x   = eval_rank_ic(va.loc[Xva.index], Y_REG, px_val)
    sprd_x, tstat_x, _ = decile_spread(va.loc[Xva.index], Y_REG, px_val)

    # NEW: sector-neutral rank-IC for XGB
    va_xgb = va.loc[Xva.index].copy()
    va_xgb = attach_sector_labels(va_xgb)
    ic_x_sec = sector_neutral_rank_ic(va_xgb, Y_REG, px_val, sector_col="sector_eval")

    # SHAP on validation (probability space)
    try:
        import warnings
        # Suppress SHAP warnings about feature_perturbation
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=UserWarning)
            # Use 'auto' to let SHAP choose the best method, or provide a small background dataset
            # For probability output, we need interventional, so provide a small sample from training
            background_sample = Xtr.sample(min(100, len(Xtr)), random_state=42) if len(Xtr) > 0 else Xtr
            explainer = shap.TreeExplainer(xgbc, model_output="probability",
                                          feature_perturbation="interventional",
                                          data=background_sample)
            sv = explainer.shap_values(Xva)

        # Handle possible [neg_class, pos_class] list from older SHAP
        if isinstance(sv, list):
            shap_val = sv[1]  # take positive class
        else:
            shap_val = sv     # array (n_samples, n_features)

        shap_df = pd.DataFrame(shap_val, index=Xva.index, columns=Xva.columns)

        # 2.1 per-feature importance (mean |SHAP|)
        imp = (
            shap_df.abs().mean(axis=0)
            .rename("mean_abs_shap")
            .reset_index()
            .rename(columns={"index":"feature"})
        )
        imp["family"] = imp["feature"].map(feature_family)

        # 2.2 family aggregation
        fam = (
            imp.groupby("family", as_index=False)["mean_abs_shap"].sum()
            .sort_values("mean_abs_shap", ascending=False)
        )

        # 2.3 "recipe" table: among top-decile predicted winners per month,
        #     show average signed SHAP by feature + average raw feature value.
        va_scored = Xva.copy()
        va_scored["date"]  = va.loc[Xva.index, "date"].values
        va_scored["score"] = px_val

        top_rows = []
        for dt, g in va_scored.groupby(va_scored["date"].dt.to_period("M")):
            if len(g) < 10:
                continue
            thr = g["score"].quantile(0.90)
            winners_idx = g.index[g["score"] >= thr]
            if len(winners_idx) == 0:
                continue
            # avg signed SHAP and avg raw feature value among winners
            mean_shap = shap_df.loc[winners_idx].mean(axis=0).rename("mean_shap")
            mean_val  = Xva.loc[winners_idx].mean(axis=0).rename("mean_val")
            tmp = pd.concat([mean_shap, mean_val], axis=1)
            tmp["date"] = pd.Period(dt).to_timestamp("M")
            top_rows.append(tmp.reset_index().rename(columns={"index":"feature"}))

        recipe = pd.concat(top_rows, ignore_index=True) if top_rows else pd.DataFrame(columns=["feature","mean_shap","mean_val","date"])

        # save summaries for this fold 
        out_dir = Path("../../reports/stocks/tables/shap")
        out_dir.mkdir(parents=True, exist_ok=True)

        # include fold info in filenames
        suffix = f"{RUN_TAG}_{train_end}_{val_year}"

        imp_path    = out_dir / f"shap_importance_{suffix}.parquet"
        fam_path    = out_dir / f"shap_families_{suffix}.parquet"
        recipe_path = out_dir / f"shap_recipe_{suffix}.parquet"

        imp.assign(train_end=train_end, val_year=val_year).to_parquet(imp_path, index=False)
        fam.assign(train_end=train_end, val_year=val_year).to_parquet(fam_path, index=False)
        recipe.assign(train_end=train_end, val_year=val_year).to_parquet(recipe_path, index=False)

        print(f"SHAP saved → {imp_path.name}, {fam_path.name}, {recipe_path.name}")

    except Exception as e:
        import traceback
        print(f"[SHAP skipped] {type(e).__name__}: {e}")
        print(f"   Traceback (first 3 lines):")
        tb_lines = traceback.format_exc().split('\n')[:4]
        for line in tb_lines:
            if line.strip():
                print(f"   {line}")

    out = {
        "train_end": train_end, "val_year": val_year,
        "logit_auc": auc_l, "logit_best_t": best_t, "logit_balacc": best_bal,
        "logit_rankIC": ic_l, "logit_rankIC_sec": ic_l_sec,
        "logit_spread": sprd_l, "logit_t": tstat_l, "months": n_m,
        "xgb_auc": auc_x, "xgb_best_t": best_tx, "xgb_balacc": best_bax,
        "xgb_rankIC": ic_x, "xgb_rankIC_sec": ic_x_sec,
        "xgb_spread": sprd_x, "xgb_t": tstat_x,
        "n_train": len(Xtr), "n_val": len(Xva), "n_feats": len(FEATS)
    }
    return out, FEATS

# Save original unfiltered dataframe for US/UK split analysis
# Always save original before any GEO filtering (needed for US/UK comparison later)
df_original = df.copy()

# Filter by GEO if specified (before CV loop)
# This ensures US and UK SHAP files are computed on separate data
if GEO == "US":
    df = df[df["country_UK"] == 0].copy()
    years = df["date"].dt.year
    print(f"\nFiltered to US-only: {len(df)} rows, {df['ticker'].nunique()} tickers")
    print(f"   Date range: {df['date'].min()} → {df['date'].max()}\n")
elif GEO == "UK":
    df = df[df["country_UK"] == 1].copy()
    years = df["date"].dt.year
    print(f"\nFiltered to UK-only: {len(df)} rows, {df['ticker'].nunique()} tickers")
    print(f"   Date range: {df['date'].min()} → {df['date'].max()}\n")
else:
    years = df["date"].dt.year
    print(f"\nRunning on Global data (all countries): {len(df)} rows, {df['ticker'].nunique()} tickers\n")

# Run the folds
results = []
last_feats = None
for f in FOLDS:
    r, feats = run_fold(f["train_end"], f["val"])
    results.append(r)
    last_feats = feats

res = pd.DataFrame(results)
print(res)
print("\nMean (validation across folds)")
print(res[["logit_auc","xgb_auc","logit_rankIC","xgb_rankIC","logit_rankIC_sec","xgb_rankIC_sec","logit_spread","xgb_spread"]].mean())

# Persist table (nice-to-have)
REPORTS = Path("../../reports/stocks/tables"); REPORTS.mkdir(parents=True, exist_ok=True)
out_path = REPORTS / f"stocks_model_cv_{DATA_VERSION}_1b.parquet"  # v2 or v3
res.to_parquet(out_path, index=False)
print(f"Saved cross-fold table → {out_path}")
print(f"Results saved for {DATA_VERSION} - change DATA_VERSION in Cell 1 to compare v2 vs v3")



Running on Global data (all countries): 22929 rows, 100 tickers

Fold 2016-2017: Before regime flags - tr=13629, va=1200
Regime flags created - dtr: ['high_vix', 'oil_shock', 'strong_dollar', 'strong_gbp', 'bear_tape', 'vol_regime', 'strong_gbp_uk']
Regime flags non-null counts - dtr: {'high_vix': 13628, 'oil_shock': 13628, 'strong_dollar': 13628, 'strong_gbp': 13628, 'bear_tape': 13628, 'vol_regime': 13628, 'strong_gbp_uk': 13628}
Regime flags non-null counts - dva: {'high_vix': 1199, 'oil_shock': 1199, 'strong_dollar': 1199, 'strong_gbp': 1199, 'bear_tape': 1199, 'vol_regime': 1199, 'strong_gbp_uk': 1199}
Dropped NaNs for regime flags: ['high_vix', 'oil_shock', 'strong_dollar', 'strong_gbp', 'bear_tape', 'vol_regime', 'strong_gbp_uk']
Fold 2016-2017: After regime flags - tr=13628, va=1199
[GEO-FEATURES] Detected: MIXED (uk_pct=0.500) → Using 6 macro features: ['vix_lvl', 'dxy_lvl', 'us10y_lvl', 'uk10y_lvl', 'dex_usuk', 'brent_d3m']
Feature filtering: 43 → 40 (removed 3 high-NaN feat

In [25]:
# US vs UK split CV

def run_cv_slice(d, label):
    """
    Run the same CV loop on a filtered dataset (US-only or UK-only).
    Temporarily replaces global df, years, GEO, and RUN_TAG, runs folds, then restores.
    """
    global df, years, GEO, RUN_TAG
    # Backup
    bak_df, bak_years, bak_geo, bak_run_tag = df, years, GEO, RUN_TAG
    # Replace with slice
    df = d.copy()
    years = df["date"].dt.year
    # Set GEO and RUN_TAG for correct SHAP file naming
    GEO = label  # "US" or "UK"
    RUN_TAG = f"{DATA_VERSION_FULL}_{GEO}" if GEO else DATA_VERSION_FULL
    rows = []
    for f in FOLDS:
        r, _ = run_fold(f["train_end"], f["val"])
        r["label"] = label
        rows.append(r)
    res = pd.DataFrame(rows)
    # Restore
    df, years, GEO, RUN_TAG = bak_df, bak_years, bak_geo, bak_run_tag
    return res

# Check if country_UK column exists

if "country_UK" in df_original.columns:
    df_us = df_original[df_original["country_UK"] == 0].copy()
    df_uk = df_original[df_original["country_UK"] == 1].copy()

    print(f"\n{'='*60}")
    print(f"US-only: {len(df_us)} rows, {df_us['ticker'].nunique()} tickers")
    print(f"UK-only: {len(df_uk)} rows, {df_uk['ticker'].nunique()} tickers")
    print(f"{'='*60}\n")

    res_us = run_cv_slice(df_us, "US")
    res_uk = run_cv_slice(df_uk, "UK")

    print("\n=== US average (XGB) ===")
    print(res_us[["xgb_auc","xgb_rankIC","xgb_rankIC_sec","xgb_spread"]].mean())

    print("\n=== UK average (XGB) ===")
    print(res_uk[["xgb_auc","xgb_rankIC","xgb_rankIC_sec","xgb_spread"]].mean())

    # Save US/UK results
    res_us.to_parquet(REPORTS / f"stocks_model_cv_{DATA_VERSION}_1b_US.parquet", index=False)
    res_uk.to_parquet(REPORTS / f"stocks_model_cv_{DATA_VERSION}_1b_UK.parquet", index=False)
    print(f"\nSaved US results → {REPORTS}/stocks_model_cv_{DATA_VERSION}_1b_US.parquet")
    print(f"Saved UK results → {REPORTS}/stocks_model_cv_{DATA_VERSION}_1b_UK.parquet")
else:
    print("'country_UK' column not found. Skipping US/UK split analysis.")



US-only: 11469 rows, 50 tickers
UK-only: 11460 rows, 50 tickers

Fold 2016-2017: Before regime flags - tr=6819, va=600
Regime flags created - dtr: ['high_vix', 'oil_shock', 'strong_dollar', 'strong_gbp', 'bear_tape', 'vol_regime', 'strong_gbp_uk']
Regime flags non-null counts - dtr: {'high_vix': 6818, 'oil_shock': 6818, 'strong_dollar': 6818, 'strong_gbp': 6818, 'bear_tape': 6818, 'vol_regime': 6818, 'strong_gbp_uk': 6818}
Regime flags non-null counts - dva: {'high_vix': 599, 'oil_shock': 599, 'strong_dollar': 599, 'strong_gbp': 599, 'bear_tape': 599, 'vol_regime': 599, 'strong_gbp_uk': 599}
Dropped NaNs for regime flags: ['high_vix', 'oil_shock', 'strong_dollar', 'strong_gbp', 'bear_tape', 'vol_regime', 'strong_gbp_uk']
Fold 2016-2017: After regime flags - tr=6818, va=599
[GEO-FEATURES] Detected: US (uk_pct=0.000) → Using 4 macro features: ['vix_lvl', 'dxy_lvl', 'us10y_lvl', 'brent_d3m']
Feature filtering: 41 → 38 (removed 3 high-NaN features)
Removed features: ['pv_ma3', 'pv_z12', '

In [26]:
print("Single-fold test ")
print([name for name in globals() if 'run_fold' in name])

test_fold = FOLDS[0]  # 2016 train, 2017 val
print(f"Testing fold: train_end={test_fold['train_end']}, val_year={test_fold['val']}")
print(f"Data shape: {df.shape} | Date range: {df['date'].min()} → {df['date'].max()}")

try:
    r, feats = run_fold(test_fold["train_end"], test_fold["val"])
    print("SUCCESS")
    print(r)
    print(f"Features used: {len(feats)}")
except Exception as e:
    import traceback
    print(f"ERROR: {e}")
    traceback.print_exc()


Single-fold smoke test 
['run_fold']
Testing fold: train_end=2016, val_year=2017
Data shape: (22929, 50) | Date range: 2005-02-28 00:00:00 → 2024-09-30 00:00:00
Fold 2016-2017: Before regime flags - tr=13629, va=1200
Regime flags created - dtr: ['high_vix', 'oil_shock', 'strong_dollar', 'strong_gbp', 'bear_tape', 'vol_regime', 'strong_gbp_uk']
Regime flags non-null counts - dtr: {'high_vix': 13628, 'oil_shock': 13628, 'strong_dollar': 13628, 'strong_gbp': 13628, 'bear_tape': 13628, 'vol_regime': 13628, 'strong_gbp_uk': 13628}
Regime flags non-null counts - dva: {'high_vix': 1199, 'oil_shock': 1199, 'strong_dollar': 1199, 'strong_gbp': 1199, 'bear_tape': 1199, 'vol_regime': 1199, 'strong_gbp_uk': 1199}
Dropped NaNs for regime flags: ['high_vix', 'oil_shock', 'strong_dollar', 'strong_gbp', 'bear_tape', 'vol_regime', 'strong_gbp_uk']
Fold 2016-2017: After regime flags - tr=13628, va=1199
[GEO-FEATURES] Detected: MIXED (uk_pct=0.500) → Using 6 macro features: ['vix_lvl', 'dxy_lvl', 'us10y_